<a href="https://colab.research.google.com/github/moeeed2006-ops/Abdul-Moeed-flyrank-ml-work/blob/main/work/capstone_content_refresh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: Install & Imports
!pip install duckdb pandas scikit-learn numpy matplotlib seaborn -q

import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_fscore_support, confusion_matrix

# Initialize DuckDB connection
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

# Set your Hugging Face Token if needed:
# con.execute("CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN 'your_hf_token');")
print("Environment ready!")

Environment ready!


In [2]:
# Cell 2: Feature Engineering Query
query = """
WITH page_metrics AS (
    SELECT
        content_id,
        -- Baseline Period
        SUM(CASE WHEN report_date < '2026-05-01' THEN clicks ELSE 0 END) AS baseline_clicks,
        SUM(CASE WHEN report_date < '2026-05-01' THEN impressions ELSE 0 END) AS baseline_impressions,
        AVG(CASE WHEN report_date < '2026-05-01' THEN average_position ELSE NULL END) AS baseline_position,

        -- Recent Period
        SUM(CASE WHEN report_date >= '2026-05-01' THEN clicks ELSE 0 END) AS recent_clicks,
        SUM(CASE WHEN report_date >= '2026-05-01' THEN impressions ELSE 0 END) AS recent_impressions,
        AVG(CASE WHEN report_date >= '2026-05-01' THEN average_position ELSE NULL END) AS recent_position
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    GROUP BY content_id
)
SELECT
    content_id,
    baseline_clicks,
    baseline_impressions,
    baseline_position,
    recent_clicks,
    recent_impressions,
    recent_position,

    -- Safe Features (derived strictly from baseline period to avoid leakage)
    (baseline_clicks / NULLIF(baseline_impressions, 0)) AS baseline_ctr,

    -- Target Label: High historical traffic (>30 clicks) with >20% decay in recent clicks
    CASE
        WHEN baseline_clicks > 30 AND ((recent_clicks - baseline_clicks) / NULLIF(baseline_clicks, 0)) < -0.20
        THEN 1
        ELSE 0
    END AS needs_refresh
FROM page_metrics
WHERE baseline_impressions > 100;
"""

# If running on local mock or Hugging Face warehouse:
# df = con.execute(query).df()

# Fallback: Synthetic Data Generator (if HF dataset is loading slow)
np.random.seed(42)
n_samples = 2500
df = pd.DataFrame({
    'content_id': [f'page_{i}' for i in range(n_samples)],
    'baseline_clicks': np.random.exponential(scale=50, size=n_samples) + 5,
    'baseline_impressions': np.random.exponential(scale=1000, size=n_samples) + 100,
    'baseline_position': np.random.uniform(1.0, 30.0, size=n_samples),
    'baseline_ctr': np.random.beta(2, 20, size=n_samples),
})
# Synthetic decay logic for demonstration
df['recent_clicks'] = df['baseline_clicks'] * np.random.uniform(0.3, 1.4, size=n_samples)
df['needs_refresh'] = ((df['baseline_clicks'] > 30) & ((df['recent_clicks'] - df['baseline_clicks']) / df['baseline_clicks'] < -0.20)).astype(int)

print(f"Dataset Loaded! Shape: {df.shape}")
print(f"Base Rate (Target %): {df['needs_refresh'].mean() * 100:.2f}%")

Dataset Loaded! Shape: (2500, 7)
Base Rate (Target %): 27.16%


In [3]:
# Cell 3: Baseline vs Machine Learning Evaluation
# Define features (deliberately excluding recent_* metrics to avoid data leakage)
features = ['baseline_clicks', 'baseline_impressions', 'baseline_position', 'baseline_ctr']
X = df[features]
y = df['needs_refresh']

# Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# 1. Rule-Based Baseline (Heuristic: Low CTR & High Impression but decaying)
y_pred_baseline = (X_test['baseline_position'] > 10.0) & (X_test['baseline_ctr'] < X_test['baseline_ctr'].median())
y_pred_baseline = y_pred_baseline.astype(int)

# 2. Random Forest Model
rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

# Base Rate
base_rate = y_test.mean()

print(f"--- BASELINE MODEL METRICS ---")
print(f"Base Rate: {base_rate:.3f}")
print(classification_report(y_test, y_pred_baseline, zero_division=0))

print(f"\n--- RANDOM FOREST MODEL METRICS ---")
print(classification_report(y_test, y_pred_rf))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_rf):.4f}")

--- BASELINE MODEL METRICS ---
Base Rate: 0.272
              precision    recall  f1-score   support

           0       0.72      0.67      0.70       364
           1       0.26      0.30      0.28       136

    accuracy                           0.57       500
   macro avg       0.49      0.49      0.49       500
weighted avg       0.59      0.57      0.58       500


--- RANDOM FOREST MODEL METRICS ---
              precision    recall  f1-score   support

           0       0.74      0.98      0.84       364
           1       0.53      0.07      0.13       136

    accuracy                           0.73       500
   macro avg       0.63      0.52      0.48       500
weighted avg       0.68      0.73      0.65       500

ROC-AUC Score: 0.7879


In [4]:
# Cell 4: Interpretability & Ranked Opportunity Engine
# Feature Importances
importances = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=False)
print("--- FEATURE IMPORTANCES ---")
print(importances)

# Add predictions to test set for Action Playbook
X_test_output = X_test.copy()
X_test_output['needs_refresh_prob'] = y_prob_rf
X_test_output['actual_refresh_needed'] = y_test

# Assign Reason Codes based on model signals
def assign_reason_code(row):
    if row['baseline_position'] > 12:
        return 'RANK_DECAY_PAGE_2'
    elif row['baseline_ctr'] < 0.02:
        return 'LOW_CTR_HIGH_IMPRESSIONS'
    elif row['baseline_clicks'] > 50:
        return 'HIGH_VALUE_TRAFFIC_LOSS'
    return 'MONITOR_ENGAGEMENT'

X_test_output['reason_code'] = X_test_output.apply(assign_reason_code, axis=1)

# Display Top Ranked Refresh Opportunities
top_ranked = X_test_output.sort_values(by='needs_refresh_prob', ascending=False).head(10)
print("\n--- TOP RANKED CONTENT REFRESH OPPORTUNITIES ---")
print(top_ranked[['needs_refresh_prob', 'reason_code', 'baseline_clicks', 'baseline_position']])

--- FEATURE IMPORTANCES ---
baseline_clicks         0.770005
baseline_impressions    0.085333
baseline_position       0.073599
baseline_ctr            0.071063
dtype: float64

--- TOP RANKED CONTENT REFRESH OPPORTUNITIES ---
      needs_refresh_prob         reason_code  baseline_clicks  \
526             0.562586   RANK_DECAY_PAGE_2        31.791747   
1127            0.538531  MONITOR_ENGAGEMENT        31.250725   
2256            0.532363  MONITOR_ENGAGEMENT        41.433159   
489             0.527321   RANK_DECAY_PAGE_2       115.483791   
2255            0.526696   RANK_DECAY_PAGE_2        47.405017   
928             0.525445   RANK_DECAY_PAGE_2        55.184867   
1250            0.525312   RANK_DECAY_PAGE_2        84.613627   
1377            0.521275   RANK_DECAY_PAGE_2        41.161493   
1917            0.519673   RANK_DECAY_PAGE_2        49.952551   
1014            0.513198   RANK_DECAY_PAGE_2        66.136578   

      baseline_position  
526           19.529928  
1127   